In [ ]:
!pip install evaluate

In [ ]:
from google.colab import drive
import json

# ၁။ Google Drive ကို Colab နဲ့ ချိတ်ဆက်ပါ (Pop-up တက်လာရင် Allow/Connect နှိပ်ပေးပါ)
drive.mount('/content/drive')

# ၂။ Drive Mount ပြီးမှ ဖိုင်လမ်းကြောင်းကို ဖတ်ပါ
file_path = "/content/drive/MyDrive/segmentation_sample_300.jsonl"

raw_data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line))

print(f"စုစုပေါင်း စာကြောင်း {len(raw_data)} ကြောင်း ဖတ်ယူပြီးပါပြီ။")

In [ ]:
import json

raw_data = []

# သင့် .jsonl ဖိုင်လမ်းကြောင်းကို ထည့်ပါ
file_path = "/content/drive/MyDrive/segmentation_sample_300.jsonl"  # သို့မဟုတ် Google Drive လမ်းကြောင်း '/content/drive/MyDrive/my_data.jsonl'

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        # စာကြောင်းအလွတ်မဟုတ်ရင် JSON အဖြစ် ပြောင်းပြီး list ထဲထည့်ပါ
        if line.strip():
            raw_data.append(json.loads(line))

print(f"စုစုပေါင်း စာကြောင်း {len(raw_data)} ကြောင်း ဖတ်ယူပြီးပါပြီ။")

စုစုပေါင်း စာကြောင်း 300 ကြောင်း ဖတ်ယူပြီးပါပြီ။


In [ ]:
# 1. Raw Data မှ Hugging Face Dataset သို့ ပြောင်းခြင်း
dataset_raw = Dataset.from_list(raw_data)

# 2. Tokenize နှင့် Alignment ပြုလုပ်ရန် function (seg_labels ဟု ပြင်ထားပါသည်)
def preprocess_function(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = examples["seg_labels"]  # 'ner_tags' အစား 'seg_labels' ဟု ပြောင်းထားသည်
    new_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

# Mapping ပြုလုပ်ခြင်း
tokenized_dataset = dataset_raw.map(preprocess_function, batched=True)

# 3. Train/Validation Dataset ခွဲခြင်း
dataset = tokenized_dataset.train_test_split(test_size=0.2)
dataset["validation"] = dataset.pop("test")

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
print(dataset_raw[0].keys())

In [ ]:
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset, DatasetDict
import evaluate

# Load pre-trained model and tokenizer (e.g., bert-base-multilingual-cased)
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Map labels for sequence labeling (e.g., BIO tagging or B/I token boundary tags)
label_list = ["O", "B-SEG", "I-SEG"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)
        elif word_id != current_word:
            current_word = word_id
            new_labels.append(labels[word_id])
        else:
            # Subwords following the first token share the label or use I-tag
            new_labels.append(labels[word_id])
    return new_labels

def preprocess_function(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [ ]:
import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset

# 1. Model & Label Mappings (Updated for ["B", "I", "O"])
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_list = ["B", "I", "O"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# 2. Align Labels Function (Converts B, I, O string labels to integer IDs)
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)
        else:
            label_val = labels[word_id]
            # Map string tags ("B", "I", "O") directly to their integer IDs
            label_id = label2id[label_val] if isinstance(label_val, str) else label_val
            new_labels.append(label_id)
    return new_labels

# 3. Preprocess Dataset
def preprocess_function(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = examples["seg_labels"]
    new_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

# Build Dataset Split
dataset_raw = Dataset.from_list(raw_data)
tokenized_dataset = dataset_raw.map(preprocess_function, batched=True)
dataset = tokenized_dataset.train_test_split(test_size=0.2)
dataset["validation"] = dataset.pop("test")

# 4. Metrics Calculation
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [p for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [l for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    flat_preds = [item for sublist in true_predictions for item in sublist]
    flat_labels = [item for sublist in true_labels for item in sublist]

    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_labels, flat_preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(flat_labels, flat_preds)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": acc,
    }

# 5. Model Setup & Training
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./burmese_segmentation_checkpoint",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Start Fine-Tuning
trainer.train()
trainer.save_model("./best_burmese_segmentation_model")
tokenizer.save_pretrained("./best_burmese_segmentation_model")

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.656918,0.649561,0.639489,0.565084,0.639489
2,No log,0.630948,0.715006,0.686640,0.639455,0.686640
3,No log,0.622103,0.661557,0.669450,0.647830,0.669450


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_burmese_segmentation_model/tokenizer_config.json',
 './best_burmese_segmentation_model/tokenizer.json')

In [ ]:
from transformers import pipeline

segmenter = pipeline(
    "token-classification",
    model="./best_burmese_segmentation_model",
    tokenizer="./best_burmese_segmentation_model"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# 1. Clean pipeline output
segmented_units = []
for entity in predictions:
    word = entity["word"].replace("##", "")
    tag = entity["entity_group"]

    if tag == "B" or not segmented_units:
        segmented_units.append(word)
    else: # Tag is 'I'
        segmented_units[-1] += word

# 2. Join segments with a delimiter (e.g., spaces or '|')
segmented_text = " | ".join(segmented_units)

print("Segmented Output:", segmented_text)

Segmented Output: မြန်မာစာလုံးပေါင်းသတ်ပုံ


In [ ]:
from transformers import pipeline

# Load your fine-tuned model and tokenizer
segmenter = pipeline(
    "token-classification",
    model="./best_burmese_segmentation_model",
    tokenizer="./best_burmese_segmentation_model",
    aggregation_strategy="simple"
)

# Test with Burmese text
sample_text = "မြန်မာစာလုံးပေါင်းသတ်ပုံ"
predictions = segmenter(sample_text)

print(predictions)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'B', 'score': np.float32(0.6451213), 'word': 'မြန်မာ', 'start': 0, 'end': 6}, {'entity_group': 'I', 'score': np.float32(0.6202408), 'word': '##စာလုံးပေါင်းသတ်ပုံ', 'start': 6, 'end': 24}]
